# Distribution shift

* **Table 2** (`tab:ood_results`): calibration error (`MAIN_METRIC`) and Brier under language, math and
  QA shift, averaged over models and shifts.
* **Appendix figures**: the average over all shifts with bootstrap confidence
  intervals (`ood_results_5_average.png`) and results per (train -> test) pair
  (`ood_results_4_{shift}_{metric}.png`).

Inputs: `results/shift_{language,math,qa}.csv` (`run_shift.py`) and
`results/basc_tp_shift.csv` (`run_basc_tp.py --setting shift`).

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from paper_utils import (DATASETS_MAP, FIGURES_DIR, MAIN_METRIC, METHOD_COLOR, METHOD_LABEL, METRIC_TEX,
                         RESULTS_DIR, TABLES_DIR, promote_semantic)

plt.style.use('seaborn-v0_8')
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})
INK  = plt.rcParams['axes.labelcolor']
GRID = plt.rcParams.get('grid.color', '#cccccc')

def style_axes(ax, xlabel=None, ylabel=None):
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    ax.grid(True, color=GRID, lw=0.7)
    ax.set_axisbelow(True)
    if xlabel is not None: ax.set_xlabel(xlabel, color=INK)
    if ylabel is not None: ax.set_ylabel(ylabel, color=INK)
    return ax

In [ ]:
# Columns identifying the (train, test) domains of each shift.
PAIR_COLS = {
    'language': ('train_langs', 'test_langs'),
    'math':     ('train_ds',    'test_ds'),
    'qa':       ('train_ds',    'test_ds'),
}
SHIFT_LABEL = {
    'language': 'Language shift  (Polymath)',
    'math':     'Math shift  (GSM8K ↔ Polymath)',
    'qa':       'QA shift  (SciQ / TriviaQA / WebQ)',
}

SHIFT_DFS = {}
for shift in PAIR_COLS:
    d = pd.read_csv(RESULTS_DIR / f'shift_{shift}.csv')
    if shift == 'qa':
        d = promote_semantic(d, by=('model', 'train_ds', 'test_ds'))
    SHIFT_DFS[shift] = d

# BASC-TP: Stage 2 on token probabilities, fit on the training domain.
tp = pd.read_csv(RESULTS_DIR / 'basc_tp_shift.csv')
tp = tp[tp['method'] == 'tp_iso_dcal'].assign(method='basc_tp')
for shift in SHIFT_DFS:
    add = tp[tp['shift'] == shift].rename(columns=dict(zip(('train', 'test'), PAIR_COLS[shift])))
    SHIFT_DFS[shift] = pd.concat([SHIFT_DFS[shift], add.drop(columns=['shift'])], ignore_index=True)

for shift, d in SHIFT_DFS.items():
    print(f'{shift:<8} rows={len(d):>5}  pairs={d[list(PAIR_COLS[shift])].drop_duplicates().shape[0]}  '
          f'models={d["model"].nunique()}')

In [ ]:
def shift_mean_sem(shift, metric):
    """Per method: mean and SEM across the (pair, model) combinations of a shift."""
    d = SHIFT_DFS[shift]
    tr, te = PAIR_COLS[shift]
    per = d.groupby(['method', 'model', tr, te])[metric].mean().reset_index()
    out = per.groupby('method')[metric].agg(mean='mean', std='std', n='count').reset_index()
    out['sem'] = out['std'] / np.sqrt(out['n'].clip(lower=1))
    return out.set_index('method')

## Table 2

In [ ]:
TABLE2_METHODS = ['logprob', 'verbal_conf_stem6', 'ridge_clip_sc_k10', 'basc_tp', 'basc_k2', 'oracle_sc_k100']
REFERENCE = {'oracle_sc_k100'}   # below the midrule; not considered for bold
SHIFT_COL_LABEL = {'language': 'Language', 'math': 'Math', 'qa': 'QA'}
METRIC_COL_LABEL = {**METRIC_TEX, 'Brier': 'Brier'}

def table2(methods, metrics=(MAIN_METRIC, 'Brier')):
    frames = []
    for shift in SHIFT_DFS:
        for metric in metrics:
            agg = shift_mean_sem(shift, metric).reindex(methods)['mean']
            agg.name = (SHIFT_COL_LABEL[shift], METRIC_COL_LABEL[metric])
            frames.append(agg)
    tbl = pd.concat(frames, axis=1)
    tbl.columns = pd.MultiIndex.from_tuples(tbl.columns)
    for metric in metrics:
        lab = METRIC_COL_LABEL[metric]
        tbl[('Average', lab)] = tbl.xs(lab, axis=1, level=1).mean(axis=1, skipna=False)
    tbl.index = [METHOD_LABEL[m] for m in tbl.index]
    return tbl

def table2_latex(tbl, methods):
    """LaTeX rows; in each column the best (lowest) single-generation method is bold."""
    single = [METHOD_LABEL[m] for m in methods if m not in REFERENCE]
    best = tbl.loc[single].round(3).min()
    lines = [' & ' + ' & '.join(rf'\multicolumn{{2}}{{c}}{{{g}}}' for g in dict.fromkeys(tbl.columns.get_level_values(0))) + r' \\',
             ' & ' + ' & '.join(tbl.columns.get_level_values(1)) + r' \\', r'\midrule']
    for m in methods:
        label = METHOD_LABEL[m]
        if m in REFERENCE:
            lines.append(r'\midrule')
        cells = [rf'\textbf{{{v:.3f}}}' if (m not in REFERENCE and round(v, 3) == best[c]) else f'{v:.3f}'
                 for c, v in tbl.loc[label].items()]
        lines.append(label + ' & ' + ' & '.join(cells) + r' \\')
    return '\n'.join(lines)

tbl2 = table2(TABLE2_METHODS)
tex2 = table2_latex(tbl2, TABLE2_METHODS)
(TABLES_DIR / 'table2_distribution_shift.tex').write_text(tex2 + '\n')
print(tex2)
tbl2.round(3)

## Appendix: average over all shifts
Point estimate = mean over the three shift families (as in Table 2's Average
column); 95% CI from a model-level cluster bootstrap within each family.

In [ ]:
AVG_METHODS = ['logprob', 'ans_logprob', 'verbal_conf_stem6', 'ridge_clip_sc_k10', 'ssc_ridge_k10',
               'basc_k2', 'basc_k5', 'basc_tp', 'oracle_sc_k100']
N_BOOT, CI_LEVEL, BOOT_SEED = 10000, 0.95, 0

def average_across_shifts(methods, metric):
    rng = np.random.default_rng(BOOT_SEED)
    units = {}
    for s in SHIFT_DFS:
        tr, te = PAIR_COLS[s]
        units[s] = (SHIFT_DFS[s].groupby(['method', 'model', tr, te])[metric].mean()
                    .reset_index().rename(columns={metric: 'value'}))
    rows = []
    for m in methods:
        # Per shift: seed-averaged (train, test) values of this method, grouped by model.
        by_shift = [units[s][units[s]['method'] == m].groupby('model')['value'].apply(np.asarray)
                    for s in SHIFT_DFS]
        point = float(np.mean([np.concatenate(g.values).mean() for g in by_shift]))
        draws = np.empty(N_BOOT)
        for b in range(N_BOOT):
            means = []
            for g in by_shift:
                pick = rng.integers(0, len(g), len(g))          # resample models
                means.append(np.concatenate([g.values[i] for i in pick]).mean())
            draws[b] = np.mean(means)
        lo, hi = np.percentile(draws, [(1 - CI_LEVEL) / 2 * 100, (1 + CI_LEVEL) / 2 * 100])
        rows.append({'method': m, 'mean': point, 'lo': float(lo), 'hi': float(hi)})
    return pd.DataFrame(rows).set_index('method')

metrics = (MAIN_METRIC, 'Brier')
fig, axes = plt.subplots(1, len(metrics), figsize=(5.4 * len(metrics), 3.6), squeeze=False)
for ax, metric in zip(axes[0], metrics):
    a = average_across_shifts(AVG_METHODS, metric)
    y = np.arange(len(a))
    err = np.vstack([a['mean'] - a['lo'], a['hi'] - a['mean']])
    ax.barh(y, a['mean'], xerr=err, color=[METHOD_COLOR[m] for m in a.index], edgecolor='white', linewidth=1.5,
            error_kw={'capsize': 3, 'lw': 1, 'alpha': 0.6, 'ecolor': INK})
    span = float(a['mean'].max() - a['mean'].min()) or 1.0
    for v, h, yi in zip(a['mean'], a['hi'], y):
        ax.text(h + 0.03 * span, yi, f'{v:.3f}', va='center', ha='left', fontsize=10, color=INK)
    style_axes(ax, xlabel=f'{METRIC_COL_LABEL[metric]} (lower is better)')
    ax.set_yticks(y)
    ax.set_yticklabels([METHOD_LABEL[m] for m in a.index], color=INK)
    ax.invert_yaxis()
    ax.set_xlim(0.0, float(a['hi'].max()) + 0.22 * span)
fig.suptitle(f'Average across all shifts  ({int(CI_LEVEL * 100)}% CI, model-level bootstrap)',
             color=INK, fontsize=13, y=0.95)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'ood_results_5_average.png')
plt.show()

## Appendix: results per (train -> test) pair

In [ ]:
PAIR_METHODS = ['logprob', 'ans_logprob', 'verbal_conf_stem6', 'ridge_clip_sc_k10', 'ssc_ridge_k10',
                'basc_k2', 'basc_k5', 'basc_tp', 'oracle_sc_k100']
SHIFT_NCOLS = {'language': 4, 'math': 2, 'qa': 3}

def _pretty_pair(shift, train, test):
    if shift in ('math', 'qa'):
        return f'{DATASETS_MAP.get(train, train)} → {DATASETS_MAP.get(test, test)}'
    return f'{train} → {test}'

def _pair_panel(ax, methods, means, xlabel, xlim, show_ylabels=True):
    y = np.arange(len(methods))
    ax.barh(y, means, color=[METHOD_COLOR[m] for m in methods], edgecolor='white', linewidth=1.2)
    pad = 0.02 * xlim[1]
    for v, yi in zip(means, y):
        if np.isfinite(v):
            ax.text(v + pad, yi, f'{v:.2f}', va='center', ha='left', fontsize=8, color=INK)
    style_axes(ax, xlabel=xlabel)
    ax.set_yticks(y)
    ax.set_yticklabels([METHOD_LABEL[m] for m in methods] if show_ylabels else [], color=INK, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlim(*xlim)

def plot_per_pair(shift, methods, metric, savepath=None):
    d = SHIFT_DFS[shift]
    tr, te = PAIR_COLS[shift]
    per = d.groupby([tr, te, 'method'])[metric].mean().unstack('method').reindex(columns=methods).sort_index()
    pairs = list(per.index)
    ncols = SHIFT_NCOLS[shift]
    nrows = math.ceil(len(pairs) / ncols)
    vmax = np.nanmax(per.values)
    xlim = (0.0, vmax * 1.28 if vmax > 0 else 1.0)
    xlabel = METRIC_COL_LABEL[metric]
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 3 * nrows + 0.3), squeeze=False, sharex=True)
    for i in range(nrows * ncols):
        ax = axes[i // ncols][i % ncols]
        if i >= len(pairs):
            ax.set_visible(False)
            continue
        pair = pairs[i]
        _pair_panel(ax, methods, per.loc[pair].reindex(methods).to_numpy(dtype=float),
                    xlabel=xlabel if (i // ncols) == nrows - 1 else '', xlim=xlim,
                    show_ylabels=(i % ncols) == 0)
        ax.set_title(_pretty_pair(shift, pair[0], pair[1]), color=INK, fontsize=11)
    fig.suptitle(f'{SHIFT_LABEL[shift]}  —  {xlabel} by (train → test)', fontsize=15, color=INK, y=0.97)
    plt.tight_layout()
    if savepath:
        fig.savefig(savepath)
    plt.show()

for shift in SHIFT_DFS:
    for metric in (MAIN_METRIC, 'Brier'):
        plot_per_pair(shift, PAIR_METHODS, metric, savepath=FIGURES_DIR / f'ood_results_4_{shift}_{metric}.png')